<a href="https://colab.research.google.com/github/dunurejoice20/electricity-forecasting./blob/main/notebooks/04_forecasting_rejoice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -q "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip" -O power.zip
!unzip -o -q power.zip

In [2]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load and make hourly
df = pd.read_csv("household_power_consumption.txt", sep=";", na_values="?", low_memory=False)
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
hourly = df.set_index("datetime").sort_index()["Global_active_power"].resample("h").mean()
hourly = hourly.interpolate(limit=6)
hourly.to_frame().reset_index().to_csv("clean_hourly.csv", index=False)
print("Hours:", len(hourly), "| still missing:", hourly.isna().sum())

# 2. Features (past values only)
d = hourly.to_frame("actual")
d["hour"] = d.index.hour
d["dayofweek"] = d.index.dayofweek
d["month"] = d.index.month
d["lag_1"] = d["actual"].shift(1)
d["lag_24"] = d["actual"].shift(24)
d["lag_168"] = d["actual"].shift(168)
d["roll_24"] = d["actual"].shift(1).rolling(24).mean()
d = d.dropna()

# 3. Time-based split: train on the past, test on the future
train = d[d.index < "2010-06-01"]
test = d[d.index >= "2010-06-01"]
feats = ["hour", "dayofweek", "month", "lag_1", "lag_24", "lag_168", "roll_24"]

def scores(name, y, p):
    return {"Model": name,
            "MAE": round(mean_absolute_error(y, p), 4),
            "RMSE": round(np.sqrt(mean_squared_error(y, p)), 4),
            "MAPE": round(np.mean(np.abs((y - p) / y)) * 100, 2)}

preds = {"Baseline (yesterday)": test["lag_24"].values}
preds["Linear Regression"] = LinearRegression().fit(train[feats], train["actual"]).predict(test[feats])
rf = RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
preds["Random Forest"] = rf.fit(train[feats], train["actual"]).predict(test[feats])

results = pd.DataFrame([scores(n, test["actual"].values, p) for n, p in preds.items()])
results.to_csv("model_comparison.csv", index=False)
print(results)

# 4. Predictions and anomalies from the best model
best = results.sort_values("MAE").iloc[0]["Model"]
out = pd.DataFrame({"actual": test["actual"], "forecast": preds[best]}, index=test.index)
resid = out["actual"] - out["forecast"]
out["is_anomaly"] = resid.abs() > 3 * resid.std()
out.reset_index().to_csv("predictions.csv", index=False)
print("Best model:", best, "| anomalies flagged:", int(out["is_anomaly"].sum()))

Hours: 34589 | still missing: 378
                  Model     MAE    RMSE   MAPE
0  Baseline (yesterday)  0.4847  0.7305  64.63
1     Linear Regression  0.3545  0.4968  51.17
2         Random Forest  0.3195  0.4619  44.94
Best model: Random Forest | anomalies flagged: 55
